# Bootstrapping Full
Esta sección repite lo visto en el modelado con la diferencia que utiliza todo el dataset y no aplica train/test con fines de analizar posibles diferencias y efecto que podría ocasionar la partición del dataset

# 1. LR Full

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_lr_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

LR_PARAMS = dict(
    solver="liblinear",
    C=1.0,
    max_iter=2000,
    random_state=BASE_SEED,
    class_weight="balanced"
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

# Usar TODO el dataset sin reset_index
X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 LR FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "lr_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"lr_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "lr_full_detalle.csv"
resumen_path = CARPETA_OUT / "lr_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ LR FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 LR FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 2. RF Full

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_rf_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

RF_PARAMS = dict(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=BASE_SEED,
    n_jobs=-1
)

MODELO = RandomForestClassifier(**RF_PARAMS)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🌲 RF FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "rf_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"rf_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "rf_full_detalle.csv"
resumen_path = CARPETA_OUT / "rf_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ RF FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌲 RF FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌲 RF FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌲 RF FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 3. DT Full

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_dt_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

DT_PARAMS = dict(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=BASE_SEED
)

MODELO = DecisionTreeClassifier(**DT_PARAMS)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🌳 DT FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "dt_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"dt_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "dt_full_detalle.csv"
resumen_path = CARPETA_OUT / "dt_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ DT FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌳 DT FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌳 DT FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌳 DT FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 4. KNN Full

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_knn_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

KNN_PARAMS = dict(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(**KNN_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

# Usar TODO el dataset sin reset_index
X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 KNN FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "knn_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"knn_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "knn_full_detalle.csv"
resumen_path = CARPETA_OUT / "knn_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1",
    "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ KNN FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 KNN FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 KNN FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 KNN FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/5

# Extra. SVM Full

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.svm import SVC

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_svm_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

SVM_PARAMS = dict(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    class_weight="balanced",
    random_state=BASE_SEED
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", SVC(**SVM_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_eval)
            roc = roc_auc_score(y_eval, y_score)
            pr = average_precision_score(y_eval, y_score)
        elif hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🌀 SVM FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "svm_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"svm_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "svm_full_detalle.csv"
resumen_path = CARPETA_OUT / "svm_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ SVM FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌀 SVM FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌀 SVM FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌀 SVM FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/5